In [220]:
import kagglehub
import pandas as pd

In [221]:
path = kagglehub.dataset_download('jasminemohamed2545/berber-english-160k-parallel-sentences-for-nlp')
print(path)

C:\Users\lassa\.cache\kagglehub\datasets\jasminemohamed2545\berber-english-160k-parallel-sentences-for-nlp\versions\1


In [222]:
df = pd.read_table('ber.txt')
df.head()

,Go.,Ṛuḥ.,CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8325200 (Yagurten)
0,Hi.,Azul.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
1,Hi.,Azul fell-ak.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
2,Hi.,Azul fell-am.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
3,Hi.,Azul fell-awen.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Azul fell-awent.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [223]:
df.dtypes

Go.                                                                                object
Ṛuḥ.                                                                               object
CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8325200 (Yagurten)    object
dtype: object

In [224]:
# renaming columns
df.columns = ['english', 'amazigh', 'license']
df.head()

,english,amazigh,license
0,Hi.,Azul.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
1,Hi.,Azul fell-ak.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
2,Hi.,Azul fell-am.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
3,Hi.,Azul fell-awen.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Azul fell-awent.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [225]:
# removing license column 
df_clean = df.drop(columns=['license'])
df_clean.head()

,english,amazigh
0,Hi.,Azul.
1,Hi.,Azul fell-ak.
2,Hi.,Azul fell-am.
3,Hi.,Azul fell-awen.
4,Hi.,Azul fell-awent.


In [226]:
# remove punctuation 
df_clean['english'] = df_clean['english'].str.replace(r'[^\w\s]', '', regex=True)
df_clean['amazigh'] = df_clean['amazigh'].str.replace(r'[^\w\s]', '', regex=True)

# remove extra spaces
df_clean['english'] = df_clean['english'].str.strip()
df_clean['amazigh'] = df_clean['amazigh'].str.strip()

Removes punctuations such as: . , ! ? : ; ' " ( ) [ ] { } - / \ @ # $ % & * + = < > _

In [227]:
df_clean.loc[20:25]
# space and punctuation successfully removed

,english,amazigh
20,Fire,ⵜⴰⴽⴰⵜ
21,Help,Abbuh
22,Help,ⴰⵡⵙ
23,Help,ⴰⵡⵙ
24,Jump,ⵏⴹⴻⵔ
25,Jump,ⵏⴹⴻⵕ


## Missing Data

In [228]:
# identifying missing data
df_clean[df_clean.isna().any(axis=1)]
# no missing values

,english,amazigh


In [229]:
df_clean.isna().sum() 
# no missing values

english    0
amazigh    0
dtype: int64

In [230]:
df_clean.dtypes

english    object
amazigh    object
dtype: object

In [231]:
# converting datatypes 'object' to 'string'
df_clean['english'] = df_clean['english'].astype('string')
df_clean['amazigh'] = df_clean['amazigh'].astype('string')
df_clean['tifinagh'] = df_clean['tifinagh'].astype('string')

KeyError: 'tifinagh'

## Duplicate Data

In [ ]:
# checking for duplicate translation pairs
df_clean.duplicated().sum()

np.int64(183)

In [ ]:
df_clean[df_clean.duplicated(keep=False)]

,english,amazigh
7,Run,Azzel
9,Run,Azzel
13,Wow,ⵜⵉⵍⵉⵍⴰ
14,Wow,ⵜⵉⵍⵉⵍⴰ
22,Help,ⴰⵡⵙ
...,...,...
143649,Tom went to the library just to see Mary,Tom iṛuḥ ɣer temkarḍit akken kan ad iẓer Mary
155548,If Tom had a lot of money hed buy that for you,Lemmer yesɛi Tom aṭas n yidrimen tili ad awent...
155549,If Tom had a lot of money hed buy that for you,Lemmer yesɛi Tom aṭas n yidrimen tili ad awent...
155554,If Tom had a lot of money hed buy that for you,Lemmer ili Tom aṭas n yidrimen tili ad awentti...


In [ ]:
# dropping duplicates & resetting index
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.reset_index(drop=True)

In [ ]:
df_clean.duplicated().sum()

np.int64(0)

In [ ]:
df_clean['english'].duplicated().sum()

np.int64(76784)

In [ ]:
df_clean[df['english'].duplicated(keep=False)].sort_values('english')
# duplicates appear because of gender markers e.g., nne-k and nne-m suffixes in amazigh

C:\Users\lassa\AppData\Local\Temp\ipykernel_26724\2070486705.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_clean[df['english'].duplicated(keep=False)].sort_values('english')


,english,amazigh
87543,100 years is called a century,Meyya n yiseggasen ttininas lqern
87542,100 years is called a century,Tawinest n yiseggasen ttininas lqern
99361,A baby craves its mothers milk,Alufan yerɣa ɣef uyefki n yemmas
56601,A baby has delicate skin,Aglim n ulufan rqiq
155798,A bad cold prevented her from attending the class,Twetitt tmesleɣt ijehden yernu ur tṛuḥ ara s a...
...,...,...
2097,Youve won,Tellummẓemt
104933,Youve worked hard this morning,Tcerweḍ tifi tanezzayta
39399,Yuck Whod buy this,Ɛeq Anwa ara dyesɣen aya
62679,Zero is a special number,Warun d amḍan uzzig


## Inconsistent Text & Typos Check

The data has a mix of Tifinagh script with Latin which is inconsistent, I will either try to convert all the Tifinagh into Latin, Latin to Tifinagh, or include both in separate columns. 

In [ ]:
df_clean.describe()

,english,amazigh
count,160699,160699
unique,84098,149264
top,Go home now,Ur yettett ara waya
freq,56,14


## Translation Variations

In [ ]:
(df_clean.groupby("english")["amazigh"]
 .nunique()
 .sort_values(ascending=False)
 .head(20))


english
Go home now                                 56
They hated you                              48
Do as you want                              45
Choose whichever you like                   42
You have lots of friends                    41
I want you to succeed                       40
They want you back                          40
You still have enough time                  38
Do you think this can work                  37
Do you think that this can work             37
You arent the only one Tom has cheated      36
Dont think Im going to let you do that      34
Have you finished writing the letter yet    33
Do they like you                            32
Youre not the only one who has a camera     32
They are pleased with your work             32
Tom just wants to be friends with you       32
Ive loved you since I first met you         32
They might be taller than you               32
Do you really want to win                   32
Name: amazigh, dtype: int64

## Current Findings

The corpus shows a high degree of translation variation, with many English sentences corresponding to multiple unique Amazigh translations. For example, “Go home now” has 56 unique Amazigh translations in the dataset.

Rather than treating these observations as duplicates, they may represent meaningful linguistic variation within the corpus.

Possible sources of variation include:

- Gender marking in Amazigh
- Singular and plural inflection
- Regional and dialectal variation across Amazigh varieties
- Lexical variation and alternative vocabulary choices

These patterns will be investigated further before additional cleaning or normalization decisions are made.


In [ ]:
df_clean[df_clean["english"] == "Go home now"][["english", "amazigh"]]

,english,amazigh
2406,Go home now,Ddu ɣer uxxam imira
2407,Go home now,Ddut ɣer uxxam imira
2408,Go home now,Ddum ɣer uxxam imira
2409,Go home now,Ddumt ɣer uxxam imira
2410,Go home now,Qqel ɣer uxxam imira
2411,Go home now,Qqlet ɣer uxxam imira
2412,Go home now,Qqlem ɣer uxxam imira
2413,Go home now,Qqlemt ɣer uxxam imira
2414,Go home now,Ṛuḥ ɣer uxxam imira
2415,Go home now,Ṛuḥet ɣer uxxam imira


## Tifinagh Script Column

In [ ]:
# creating a Tifinagh script column
df_clean['tifinagh'] = pd.NA
df_clean.head()

,english,amazigh,tifinagh
0,Hi,Azul,<NA>
1,Hi,Azul fellak,<NA>
2,Hi,Azul fellam,<NA>
3,Hi,Azul fellawen,<NA>
4,Hi,Azul fellawent,<NA>


In [ ]:
# detect tifinagh characters: unicode U+2D30–U+2D7F
tifinagh_mask = df_clean['amazigh'].str.contains(r'[\u2D30-\u2D7F]'
                ,regex=True,na=False)

In [ ]:
# copy tifinagh entries
df_clean.loc[tifinagh_mask, 'tifinagh'] = df_clean.loc[tifinagh_mask, 'amazigh']

# remove tifinagh entries from amazigh
df_clean.loc[tifinagh_mask, 'amazigh'] = pd.NA

# classify writing system
df_clean['script'] = 'Latin'
df_clean.loc[tifinagh_mask, 'script'] = 'Tifinagh'

df_clean[:15]

,english,amazigh,tifinagh,script
0,Hi,Azul,<NA>,Latin
1,Hi,Azul fellak,<NA>,Latin
2,Hi,Azul fellam,<NA>,Latin
3,Hi,Azul fellawen,<NA>,Latin
4,Hi,Azul fellawent,<NA>,Latin
5,Hi,<NA>,ⴰⵣⵓⵍ,Tifinagh
6,Hi,azul,<NA>,Latin
7,Run,Azzel,<NA>,Latin
8,Run,<NA>,ⴰⵣⵣⵍ,Tifinagh
9,Run,Azzel,<NA>,Latin


In [ ]:
# check whether the amazigh output is in Latin or Tifinagh 
df_clean['script'].value_counts()

script
Latin       160825
Tifinagh        57
Name: count, dtype: int64

In [ ]:
# remove missing values (NaN, None) with .dropna()
df_clean[['english', 'tifinagh']].dropna()

,english,tifinagh
5,Hi,ⴰⵣⵓⵍ
8,Run,ⴰⵣⵣⵍ
13,Wow,ⵜⵉⵍⵉⵍⴰ
14,Wow,ⵜⵉⵍⵉⵍⴰ
20,Fire,ⵜⴰⴽⴰⵜ
22,Help,ⴰⵡⵙ
23,Help,ⴰⵡⵙ
24,Jump,ⵏⴹⴻⵔ
25,Jump,ⵏⴹⴻⵕ
28,Stop,ⴱⵉⴷⴷ


row 135529 has a mix of tifinagh and latin concatenated together -> will separate the two

In [232]:
df_clean.loc[135529]

english               He doesnt know anything about politics
amazigh    Ur issin awed ḥaḥ g tsertit ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ...
Name: 135529, dtype: string

In [233]:
df_clean.loc[135529, 'amazigh'] = 'Ur issin awed ḥaḥ g tsertit'
df_clean.loc[135529, 'tifinagh'] = 'ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ'

In [235]:
df_clean.loc[135529]

english     He doesnt know anything about politics
amazigh                Ur issin awed ḥaḥ g tsertit
tifinagh                ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ
Name: 135529, dtype: object

In [237]:
df_clean[['english', 'amazigh', 'tifinagh']].dropna()

,english,amazigh,tifinagh
135529,He doesnt know anything about politics,Ur issin awed ḥaḥ g tsertit,ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ


In [238]:
df_clean[['english', 'tifinagh']].dropna()

,english,tifinagh
135529,He doesnt know anything about politics,ⵓⵔ ⵉⵙⵙⵉⵏ ⴰⵡⴷ ⵃⴰⵃ ⴳ ⵜⵙⴻⵔⵜⵉⵜ


### Findings
- Majority of amazigh data is written in Latin, Tifinagh is scarce 
- Solution: either manually add Tifinagh equivalent or focus only on Latin version